# 02 – Model Training & Evaluation

Time-aware split (train = past, validation = middle, test = most recent). No shuffling.

In [ ]:
import sys; sys.path.insert(0, '..')
import os; os.chdir('..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score, RocCurveDisplay, PrecisionRecallDisplay)
from src.preprocessing import load_raw, clean
from src.features import build_features, FEATURES, TARGET
from src.train import time_split, best_f1_threshold, evaluate

raw = time_split(clean(load_raw('data/raw/grid_data.csv')))
peak = raw.loc[raw.split == 'train', 'demand_mw'].quantile(0.90)
df = build_features(raw, peak_threshold=peak)
tr, va, te = (df[df.split == s] for s in ('train', 'val', 'test'))
for n, d in zip(('train', 'val', 'test'), (tr, va, te)):
    print(f'{n:5s} {d.timestamp.min().date()} -> {d.timestamp.max().date()}  rows={len(d):,}  event rate={d[TARGET].mean():.2%}')

## Baselines: Logistic Regression and Random Forest

In [ ]:
scaler = StandardScaler().fit(tr[FEATURES])
lr = LogisticRegression(max_iter=2000).fit(scaler.transform(tr[FEATURES]), tr[TARGET])
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, max_depth=14, n_jobs=-1, random_state=42).fit(tr[FEATURES], tr[TARGET])

def proba(name, X): return lr.predict_proba(scaler.transform(X[FEATURES]))[:, 1] if name == 'lr' else rf.predict_proba(X[FEATURES])[:, 1]

rows = {}
for name in ('lr', 'rf'):
    thr = best_f1_threshold(va[TARGET], proba(name, va))   # threshold tuned on validation only
    rows[name] = evaluate(te[TARGET], proba(name, te), thr)
pd.DataFrame(rows).T.round(3)

### Optional: XGBoost
Install with `pip install xgboost` and run the cell below.

In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42).fit(tr[FEATURES], tr[TARGET])
    p_va, p_te = xgb.predict_proba(va[FEATURES])[:, 1], xgb.predict_proba(te[FEATURES])[:, 1]
    print(pd.Series(evaluate(te[TARGET], p_te, best_f1_threshold(va[TARGET], p_va))).round(3))
except ImportError:
    print('xgboost not installed - skipping')

## Curves & confusion matrix (best-F1 threshold from validation)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, label in (('lr', 'Logistic'), ('rf', 'Random Forest')):
    RocCurveDisplay.from_predictions(te[TARGET], proba(name, te), name=label, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(te[TARGET], proba(name, te), name=label, ax=ax[1])
plt.tight_layout(); plt.show()

thr = rows['rf']['threshold']
pred = (proba('rf', te) >= thr).astype(int)
print('Random Forest @ threshold', round(thr, 3)); print(confusion_matrix(te[TARGET], pred)); print(classification_report(te[TARGET], pred, zero_division=0))

## Feature importance (Random Forest, global – not causal)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.tail(12).plot(kind='barh', figsize=(8, 5), title='Random Forest impurity importance'); plt.show()

## Save models
Run `python -m src.train` from the repo root – it trains the same models and saves them to `models/` together with metrics and importance files used by the dashboard.